# Retrieval Error Analysis

This notebook performs offline error analysis on the frozen retrieval evaluation produced by the controlled SOP / knowledge-base retrieval pipeline.

## Objectives

- Confirm the integrity of the committed retrieval evaluation artifacts.
- Compare semantic retrieval with the cross-encoder reranker at query level.
- Identify ranking improvements, regressions, unchanged results, and persistent misses.
- Classify retrieval failures using a transparent error taxonomy.
- Assess whether failures are associated with candidate retrieval, reranking, query ambiguity, corpus overlap, chunk boundaries, or relevance labels.
- Produce reproducible retrieval error-analysis artifacts.
- Recommend the retrieval configuration for the final workflow based on evidence.

## Evaluation boundary

This notebook:

- uses only the committed frozen retrieval artifacts,
- does not call OpenAI or any embedding model,
- does not execute the cross-encoder model,
- does not rebuild the FAISS index,
- does not modify the approved knowledge-base corpus,
- does not alter deterministic claim triage,
- does not use held-out claim data.

Retrieval remains non-authoritative and analyst-facing only.

## 1. Environment and Repository Setup

Resolve the repository root and load the libraries required for offline analysis.

In [1]:
from pathlib import Path
import ast
import json
import math
import sys

import pandas as pd


def find_project_root(start_path: Path) -> Path:
    """Locate the repository root using the expected project folders."""
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src, data, and notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage
Python executable: /opt/anaconda3/envs/dpclaims/bin/python
Python version: 3.11.15


## 2. Frozen Evaluation Inputs

Load the committed query set, retrieval results, summary metrics, family metrics, and evaluation manifests.

No retrieval or model inference is performed.

In [2]:
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "evaluation" / "retrieval"

ARTIFACT_PATHS = {
    "evaluation_queries": RETRIEVAL_DIR / "rag_retrieval_eval_v1.csv",
    "evaluation_query_manifest": (
        RETRIEVAL_DIR / "rag_retrieval_eval_v1_manifest.json"
    ),
    "summary_metrics": (
        RETRIEVAL_DIR
        / "retrieval_summary_metrics_with_reranker_v1.csv"
    ),
    "family_metrics": (
        RETRIEVAL_DIR
        / "retrieval_family_metrics_with_reranker_v1.csv"
    ),
    "per_query_results": (
        RETRIEVAL_DIR
        / "retrieval_per_query_results_with_reranker_v1.csv"
    ),
    "retrieval_manifest": (
        RETRIEVAL_DIR
        / "retrieval_evaluation_with_reranker_manifest_v1.json"
    ),
}

for artifact_name, artifact_path in ARTIFACT_PATHS.items():
    assert artifact_path.exists(), (
        f"Missing required artifact: {artifact_name} -> {artifact_path}"
    )

    print(
        "PASS:",
        artifact_name,
        "->",
        artifact_path.relative_to(PROJECT_ROOT),
    )

PASS: evaluation_queries -> data/evaluation/retrieval/rag_retrieval_eval_v1.csv
PASS: evaluation_query_manifest -> data/evaluation/retrieval/rag_retrieval_eval_v1_manifest.json
PASS: summary_metrics -> data/evaluation/retrieval/retrieval_summary_metrics_with_reranker_v1.csv
PASS: family_metrics -> data/evaluation/retrieval/retrieval_family_metrics_with_reranker_v1.csv
PASS: per_query_results -> data/evaluation/retrieval/retrieval_per_query_results_with_reranker_v1.csv
PASS: retrieval_manifest -> data/evaluation/retrieval/retrieval_evaluation_with_reranker_manifest_v1.json


In [3]:
evaluation_queries = pd.read_csv(
    ARTIFACT_PATHS["evaluation_queries"]
)

summary_metrics = pd.read_csv(
    ARTIFACT_PATHS["summary_metrics"]
)

family_metrics = pd.read_csv(
    ARTIFACT_PATHS["family_metrics"]
)

per_query_results = pd.read_csv(
    ARTIFACT_PATHS["per_query_results"]
)

with ARTIFACT_PATHS["evaluation_query_manifest"].open(
    "r",
    encoding="utf-8",
) as file:
    evaluation_query_manifest = json.load(file)

with ARTIFACT_PATHS["retrieval_manifest"].open(
    "r",
    encoding="utf-8",
) as file:
    retrieval_manifest = json.load(file)

print("Evaluation queries:", evaluation_queries.shape)
print("Summary metrics:", summary_metrics.shape)
print("Family metrics:", family_metrics.shape)
print("Per-query results:", per_query_results.shape)

Evaluation queries: (14, 5)
Summary metrics: (4, 7)
Family metrics: (56, 6)
Per-query results: (56, 10)


## 3. Evaluation Integrity Checks

Confirm that all four retrieval methods were evaluated against the same frozen query set and that the committed result structure is internally consistent.

In [4]:
EXPECTED_QUERY_COUNT = 14
EXPECTED_TOP_K = 3
EXPECTED_RERANKER_CANDIDATE_K = 5

EXPECTED_METHODS = {
    "Lexical TF-IDF",
    "Semantic Embedding",
    "Hybrid RRF",
    "Semantic + Cross-Encoder Reranker",
}

REQUIRED_PER_QUERY_COLUMNS = {
    "method_name",
    "query_id",
    "query_family",
    "relevant_chunk_ids",
    "retrieved_chunk_ids",
    "first_relevant_rank",
    "hit_at_1",
    "hit_at_3",
    "mrr_at_3",
    "retrieval_status",
}


def parse_serialized_list(value: object) -> list[str]:
    """Parse the Python-style list values stored in the result CSV."""
    if pd.isna(value):
        return []

    parsed_value = ast.literal_eval(str(value))

    if not isinstance(parsed_value, list):
        raise ValueError(
            f"Expected a serialized list but received: {value!r}"
        )

    return [str(item) for item in parsed_value]


missing_columns = REQUIRED_PER_QUERY_COLUMNS.difference(
    per_query_results.columns
)

assert not missing_columns, (
    "Per-query results are missing columns: "
    + ", ".join(sorted(missing_columns))
)

per_query_results = per_query_results.copy()

per_query_results["relevant_chunk_ids_list"] = (
    per_query_results["relevant_chunk_ids"].apply(
        parse_serialized_list
    )
)

per_query_results["retrieved_chunk_ids_list"] = (
    per_query_results["retrieved_chunk_ids"].apply(
        parse_serialized_list
    )
)

observed_methods = set(
    per_query_results["method_name"].dropna().unique()
)

evaluation_query_ids = set(evaluation_queries["query_id"])
result_query_ids = set(per_query_results["query_id"])

assert evaluation_queries["query_id"].nunique() == EXPECTED_QUERY_COUNT
assert per_query_results["query_id"].nunique() == EXPECTED_QUERY_COUNT
assert evaluation_query_ids == result_query_ids

assert observed_methods == EXPECTED_METHODS

assert len(per_query_results) == (
    EXPECTED_QUERY_COUNT * len(EXPECTED_METHODS)
)

assert per_query_results["retrieval_status"].eq(
    "RESULTS_FOUND"
).all()

assert per_query_results["relevant_chunk_ids_list"].apply(
    len
).ge(1).all()

assert per_query_results["retrieved_chunk_ids_list"].apply(
    len
).eq(EXPECTED_TOP_K).all()

assert evaluation_query_manifest["query_count"] == EXPECTED_QUERY_COUNT
assert (
    evaluation_query_manifest["corpus_chunk_count"]
    == retrieval_manifest["corpus"]["chunk_count"]
)

assert (
    retrieval_manifest["evaluation_set"]["query_count"]
    == EXPECTED_QUERY_COUNT
)

assert (
    retrieval_manifest["evaluation_set"]["top_k"]
    == EXPECTED_TOP_K
)

assert (
    retrieval_manifest["evaluation_set"]["reranker_candidate_k"]
    == EXPECTED_RERANKER_CANDIDATE_K
)

print("PASS: frozen evaluation queries =", EXPECTED_QUERY_COUNT)
print("PASS: retrieval methods =", len(EXPECTED_METHODS))
print("PASS: per-query result rows =", len(per_query_results))
print("PASS: top_k =", EXPECTED_TOP_K)
print(
    "PASS: reranker candidate_k =",
    EXPECTED_RERANKER_CANDIDATE_K,
)
print(
    "PASS: approved corpus chunks =",
    retrieval_manifest["corpus"]["chunk_count"],
)
print("PASS: all retrieval calls returned results")

PASS: frozen evaluation queries = 14
PASS: retrieval methods = 4
PASS: per-query result rows = 56
PASS: top_k = 3
PASS: reranker candidate_k = 5
PASS: approved corpus chunks = 21
PASS: all retrieval calls returned results


In [5]:
# Recalculate the rank metrics to validate the committed result rows.
def calculate_hit_at_1(rank: object) -> int:
    if pd.isna(rank):
        return 0

    return int(int(rank) == 1)


def calculate_hit_at_3(rank: object) -> int:
    if pd.isna(rank):
        return 0

    return int(1 <= int(rank) <= 3)


def calculate_mrr_at_3(rank: object) -> float:
    if pd.isna(rank) or int(rank) > 3:
        return 0.0

    return 1.0 / int(rank)


for row in per_query_results.itertuples(index=False):
    assert row.hit_at_1 == calculate_hit_at_1(
        row.first_relevant_rank
    )

    assert row.hit_at_3 == calculate_hit_at_3(
        row.first_relevant_rank
    )

    assert math.isclose(
        float(row.mrr_at_3),
        calculate_mrr_at_3(row.first_relevant_rank),
        rel_tol=1e-5,
        abs_tol=1e-5,
    )

print("PASS: Hit@1 values match the recorded ranks")
print("PASS: Hit@3 values match the recorded ranks")
print("PASS: MRR@3 values match the recorded ranks")

PASS: Hit@1 values match the recorded ranks
PASS: Hit@3 values match the recorded ranks
PASS: MRR@3 values match the recorded ranks


## 4. Frozen Headline Results

Display the committed aggregate metrics before conducting query-level error analysis.

In [6]:
summary_display = summary_metrics.copy()

for column in [
    "hit_at_1",
    "hit_at_3",
    "no_result_rate",
]:
    summary_display[column] = summary_display[column].map(
        lambda value: f"{value:.1%}"
    )

summary_display["mrr_at_3"] = summary_display["mrr_at_3"].map(
    lambda value: f"{value:.3f}"
)

display(
    summary_display[
        [
            "method_name",
            "query_count",
            "top_k",
            "hit_at_1",
            "hit_at_3",
            "mrr_at_3",
            "no_result_rate",
        ]
    ]
)

,method_name,query_count,top_k,hit_at_1,hit_at_3,mrr_at_3,no_result_rate
0,Lexical TF-IDF,14,3,57.1%,85.7%,0.702,0.0%
1,Semantic Embedding,14,3,78.6%,92.9%,0.857,0.0%
2,Hybrid RRF,14,3,71.4%,92.9%,0.798,0.0%
3,Semantic + Cross-Encoder Reranker,14,3,78.6%,92.9%,0.845,0.0%


## 5. Semantic Retrieval Versus Cross-Encoder Reranking

Compare the semantic baseline and reranked results at query level.

For ranking comparisons, a top-3 miss is assigned an effective rank of 4. This allows improvements and regressions to be calculated consistently without implying that the relevant chunk was actually retrieved at rank 4.

In [7]:
# Build a query-level comparison between semantic retrieval and reranking.
SEMANTIC_METHOD = "Semantic Embedding"
RERANKER_METHOD = "Semantic + Cross-Encoder Reranker"


semantic_results = (
    per_query_results.loc[
        per_query_results["method_name"].eq(SEMANTIC_METHOD),
        [
            "query_id",
            "relevant_chunk_ids_list",
            "retrieved_chunk_ids_list",
            "first_relevant_rank",
            "hit_at_1",
            "hit_at_3",
            "mrr_at_3",
        ],
    ]
    .rename(
        columns={
            "relevant_chunk_ids_list": "relevant_chunk_ids",
            "retrieved_chunk_ids_list": "semantic_retrieved_chunk_ids",
            "first_relevant_rank": "semantic_rank",
            "hit_at_1": "semantic_hit_at_1",
            "hit_at_3": "semantic_hit_at_3",
            "mrr_at_3": "semantic_mrr_at_3",
        }
    )
)

reranker_results = (
    per_query_results.loc[
        per_query_results["method_name"].eq(RERANKER_METHOD),
        [
            "query_id",
            "retrieved_chunk_ids_list",
            "first_relevant_rank",
            "hit_at_1",
            "hit_at_3",
            "mrr_at_3",
        ],
    ]
    .rename(
        columns={
            "retrieved_chunk_ids_list": "reranker_retrieved_chunk_ids",
            "first_relevant_rank": "reranker_rank",
            "hit_at_1": "reranker_hit_at_1",
            "hit_at_3": "reranker_hit_at_3",
            "mrr_at_3": "reranker_mrr_at_3",
        }
    )
)

query_comparison = (
    evaluation_queries[
        [
            "query_id",
            "query_family",
            "query_text",
            "rationale",
        ]
    ]
    .merge(
        semantic_results,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        reranker_results,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )
)


def effective_rank(rank: object) -> int:
    """Assign top-k misses a bounded comparison rank of top_k + 1."""
    if pd.isna(rank):
        return EXPECTED_TOP_K + 1

    return int(rank)


def classify_rank_change(row: pd.Series) -> str:
    """Classify the reranker's query-level effect."""
    semantic_missing = pd.isna(row["semantic_rank"])
    reranker_missing = pd.isna(row["reranker_rank"])

    if semantic_missing and reranker_missing:
        return "PERSISTENT_TOP_3_MISS"

    if row["rank_change"] > 0:
        return "IMPROVED"

    if row["rank_change"] < 0:
        return "REGRESSED"

    if (
        row["semantic_effective_rank"] == 1
        and row["reranker_effective_rank"] == 1
    ):
        return "UNCHANGED_TOP_1"

    return "UNCHANGED_RANK"


query_comparison["semantic_effective_rank"] = (
    query_comparison["semantic_rank"].apply(effective_rank)
)

query_comparison["reranker_effective_rank"] = (
    query_comparison["reranker_rank"].apply(effective_rank)
)

# Positive values mean that the reranker improved the rank.
query_comparison["rank_change"] = (
    query_comparison["semantic_effective_rank"]
    - query_comparison["reranker_effective_rank"]
)

query_comparison["mrr_change"] = (
    query_comparison["reranker_mrr_at_3"]
    - query_comparison["semantic_mrr_at_3"]
)

query_comparison["comparison_outcome"] = query_comparison.apply(
    classify_rank_change,
    axis=1,
)

query_comparison["chunks_promoted_into_top_3"] = (
    query_comparison.apply(
        lambda row: [
            chunk_id
            for chunk_id in row["reranker_retrieved_chunk_ids"]
            if chunk_id not in row["semantic_retrieved_chunk_ids"]
        ],
        axis=1,
    )
)

query_comparison["chunks_removed_from_top_3"] = (
    query_comparison.apply(
        lambda row: [
            chunk_id
            for chunk_id in row["semantic_retrieved_chunk_ids"]
            if chunk_id not in row["reranker_retrieved_chunk_ids"]
        ],
        axis=1,
    )
)

assert len(query_comparison) == EXPECTED_QUERY_COUNT
assert query_comparison["query_id"].nunique() == EXPECTED_QUERY_COUNT

print("PASS: semantic and reranker rows matched for all queries")
print("PASS: query-level rank changes calculated")
print("PASS: top-3 membership changes calculated")

PASS: semantic and reranker rows matched for all queries
PASS: query-level rank changes calculated
PASS: top-3 membership changes calculated


In [8]:
# Summarise reranker improvements, regressions, unchanged results, and misses.
outcome_order = [
    "IMPROVED",
    "REGRESSED",
    "UNCHANGED_TOP_1",
    "UNCHANGED_RANK",
    "PERSISTENT_TOP_3_MISS",
]

outcome_summary = (
    query_comparison["comparison_outcome"]
    .value_counts()
    .reindex(outcome_order, fill_value=0)
    .rename_axis("comparison_outcome")
    .reset_index(name="query_count")
)

display(outcome_summary)

changed_or_missed_queries = query_comparison.loc[
    query_comparison["comparison_outcome"].ne("UNCHANGED_TOP_1"),
    [
        "query_id",
        "query_family",
        "relevant_chunk_ids",
        "semantic_rank",
        "reranker_rank",
        "rank_change",
        "mrr_change",
        "comparison_outcome",
    ],
].copy()

display(changed_or_missed_queries)

print(
    "Aggregate MRR@3 change:",
    round(query_comparison["mrr_change"].mean(), 6),
)

,comparison_outcome,query_count
0,IMPROVED,2
1,REGRESSED,2
2,UNCHANGED_TOP_1,9
3,UNCHANGED_RANK,0
4,PERSISTENT_TOP_3_MISS,1


,query_id,query_family,relevant_chunk_ids,semantic_rank,reranker_rank,rank_change,mrr_change,comparison_outcome
3,RAG-EVAL-004,EVIDENCE_PRINCIPLE,[KB-002::S02],2.0,1.0,1,0.500000,IMPROVED
4,RAG-EVAL-005,MANUAL_REVIEW_PACKET,[KB-004::S03],1.0,3.0,-2,-0.666667,REGRESSED
7,RAG-EVAL-008,SOURCE_PRECEDENCE,[KB-001::S03],1.0,2.0,-1,-0.500000,REGRESSED
8,RAG-EVAL-009,MATERIAL_FOLLOW_UP,[KB-001::S05],2.0,1.0,1,0.500000,IMPROVED
11,RAG-EVAL-012,AUDIT_FIELDS,[KB-005::S03],NaN,NaN,0,0.000000,PERSISTENT_TOP_3_MISS


Aggregate MRR@3 change: -0.011905


## 6. Cross-Method Diagnostic Matrix

Compare all four retrieval methods for queries where the semantic baseline and reranker did not both return the relevant chunk at rank 1.

This helps distinguish isolated reranker effects from broader retrieval weaknesses.

In [9]:
# Compare the relevant-chunk rank across all four retrieval methods.
rank_matrix = (
    per_query_results.pivot(
        index="query_id",
        columns="method_name",
        values="first_relevant_rank",
    )
    .reset_index()
    .rename(
        columns={
            "Lexical TF-IDF": "lexical_rank",
            "Semantic Embedding": "semantic_rank",
            "Hybrid RRF": "hybrid_rank",
            "Semantic + Cross-Encoder Reranker": "reranker_rank",
        }
    )
)

diagnostic_matrix = (
    evaluation_queries[
        [
            "query_id",
            "query_family",
            "query_text",
        ]
    ]
    .merge(
        rank_matrix,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )
)

diagnostic_matrix = diagnostic_matrix.loc[
    ~(
        diagnostic_matrix["semantic_rank"].eq(1)
        & diagnostic_matrix["reranker_rank"].eq(1)
    )
].copy()


def display_rank(rank: object) -> object:
    """Display top-3 misses clearly without changing stored results."""
    if pd.isna(rank):
        return "MISS"

    return int(rank)


rank_columns = [
    "lexical_rank",
    "semantic_rank",
    "hybrid_rank",
    "reranker_rank",
]

diagnostic_display = diagnostic_matrix.copy()

for column in rank_columns:
    diagnostic_display[column] = diagnostic_display[column].apply(
        display_rank
    )

display(
    diagnostic_display[
        [
            "query_id",
            "query_family",
            "lexical_rank",
            "semantic_rank",
            "hybrid_rank",
            "reranker_rank",
            "query_text",
        ]
    ]
)

,query_id,query_family,lexical_rank,semantic_rank,hybrid_rank,reranker_rank,query_text
3,RAG-EVAL-004,EVIDENCE_PRINCIPLE,MISS,2,3,1,Can a supporting repair quotation make an othe...
4,RAG-EVAL-005,MANUAL_REVIEW_PACKET,MISS,1,3,3,What details must accompany a case assigned fo...
7,RAG-EVAL-008,SOURCE_PRECEDENCE,2,1,2,2,"When structured sources disagree, which source..."
8,RAG-EVAL-009,MATERIAL_FOLLOW_UP,1,2,1,1,Should the system request additional paperwork...
11,RAG-EVAL-012,AUDIT_FIELDS,3,MISS,MISS,MISS,Which fields must be recorded to make a triage...


## 7. Retrieval Error Taxonomy

Document a transparent human diagnostic assessment for the five queries affected by improvement, regression, or a persistent top-3 miss.

The assessment uses:

- frozen per-query rankings,
- retrieved chunk IDs,
- the labelled relevant section,
- the approved knowledge-base content.

The taxonomy does not modify relevance labels or retrieval outputs.

In [10]:
# Record evidence-based diagnoses for the five affected queries.
diagnostic_records = [
    {
        "query_id": "RAG-EVAL-004",
        "observed_outcome": "IMPROVED",
        "primary_error_type": (
            "SEMANTIC_NEIGHBOUR_SECTION_COMPETITION"
        ),
        "diagnostic_confidence": "HIGH",
        "evidence_summary": (
            "The semantic baseline ranked the relevant evidence-principle "
            "section second behind a neighbouring evidence-profile section. "
            "The reranker promoted the labelled section to rank 1."
        ),
        "chunk_boundary_assessment": (
            "NO_EVIDENCE_OF_BOUNDARY_FAILURE"
        ),
        "relevance_label_assessment": "LABEL_SUPPORTED",
        "bounded_recommendation": (
            "Retain as evidence that reranking can correct local section "
            "competition; no corpus change is justified."
        ),
    },
    {
        "query_id": "RAG-EVAL-005",
        "observed_outcome": "REGRESSED",
        "primary_error_type": (
            "RERANKER_WITHIN_DOCUMENT_MISORDERING"
        ),
        "diagnostic_confidence": "HIGH",
        "evidence_summary": (
            "The semantic baseline placed the required escalation-packet "
            "section at rank 1. The reranker moved two sections from the "
            "same manual-review document above it and placed it at rank 3."
        ),
        "chunk_boundary_assessment": (
            "NO_EVIDENCE_OF_BOUNDARY_FAILURE"
        ),
        "relevance_label_assessment": "LABEL_SUPPORTED",
        "bounded_recommendation": (
            "Treat this as a reranker regression and avoid assuming that "
            "reranking is always preferable."
        ),
    },
    {
        "query_id": "RAG-EVAL-008",
        "observed_outcome": "REGRESSED",
        "primary_error_type": (
            "RERANKER_NEAR_NEIGHBOUR_MISORDERING"
        ),
        "diagnostic_confidence": "HIGH",
        "evidence_summary": (
            "The semantic baseline ranked the authoritative-source-order "
            "section first. The reranker retained it in the top 3 but "
            "moved a neighbouring SOP section above it."
        ),
        "chunk_boundary_assessment": (
            "NO_EVIDENCE_OF_BOUNDARY_FAILURE"
        ),
        "relevance_label_assessment": "LABEL_SUPPORTED",
        "bounded_recommendation": (
            "Prefer the semantic baseline as the default configuration "
            "unless broader evaluation demonstrates a reranker benefit."
        ),
    },
    {
        "query_id": "RAG-EVAL-009",
        "observed_outcome": "IMPROVED",
        "primary_error_type": (
            "CROSS_DOCUMENT_SEMANTIC_COMPETITION_CORRECTED"
        ),
        "diagnostic_confidence": "HIGH",
        "evidence_summary": (
            "The semantic baseline ranked an unrelated disposition-wording "
            "section above the material-follow-up rule. The reranker "
            "promoted the labelled SOP section from rank 2 to rank 1."
        ),
        "chunk_boundary_assessment": (
            "NO_EVIDENCE_OF_BOUNDARY_FAILURE"
        ),
        "relevance_label_assessment": "LABEL_SUPPORTED",
        "bounded_recommendation": (
            "Retain as evidence that reranking can correct cross-document "
            "semantic competition."
        ),
    },
    {
        "query_id": "RAG-EVAL-012",
        "observed_outcome": "PERSISTENT_TOP_3_MISS",
        "primary_error_type": "SEMANTIC_TOP_3_RECALL_FAILURE",
        "diagnostic_confidence": "MEDIUM",
        "evidence_summary": (
            "The labelled required-result-fields section was found by "
            "lexical retrieval at rank 3 but missed by semantic, hybrid, "
            "and reranked top-3 results."
        ),
        "chunk_boundary_assessment": (
            "NO_DIRECT_EVIDENCE_OF_BOUNDARY_FAILURE"
        ),
        "relevance_label_assessment": "LABEL_SUPPORTED",
        "bounded_recommendation": (
            "Do not change chunking based on one query. Record the "
            "candidate-stage uncertainty and retain semantic retrieval "
            "as the evidence-supported default."
        ),
    },
]

retrieval_error_taxonomy = pd.DataFrame(diagnostic_records)

observed_outcomes = query_comparison[
    [
        "query_id",
        "comparison_outcome",
    ]
].rename(
    columns={
        "comparison_outcome": "calculated_outcome",
    }
)

retrieval_error_taxonomy = retrieval_error_taxonomy.merge(
    observed_outcomes,
    on="query_id",
    how="left",
    validate="one_to_one",
)

assert retrieval_error_taxonomy["calculated_outcome"].notna().all()

assert retrieval_error_taxonomy[
    "observed_outcome"
].eq(
    retrieval_error_taxonomy["calculated_outcome"]
).all()

expected_diagnostic_ids = set(
    query_comparison.loc[
        query_comparison["comparison_outcome"].ne(
            "UNCHANGED_TOP_1"
        ),
        "query_id",
    ]
)

assert set(retrieval_error_taxonomy["query_id"]) == (
    expected_diagnostic_ids
)

print("PASS: all affected queries have a diagnostic record")
print("PASS: documented outcomes match calculated outcomes")

display(
    retrieval_error_taxonomy[
        [
            "query_id",
            "observed_outcome",
            "primary_error_type",
            "diagnostic_confidence",
            "chunk_boundary_assessment",
            "relevance_label_assessment",
            "bounded_recommendation",
        ]
    ]
)

PASS: all affected queries have a diagnostic record
PASS: documented outcomes match calculated outcomes


,query_id,observed_outcome,primary_error_type,diagnostic_confidence,chunk_boundary_assessment,relevance_label_assessment,bounded_recommendation
0,RAG-EVAL-004,IMPROVED,SEMANTIC_NEIGHBOUR_SECTION_COMPETITION,HIGH,NO_EVIDENCE_OF_BOUNDARY_FAILURE,LABEL_SUPPORTED,Retain as evidence that reranking can correct ...
1,RAG-EVAL-005,REGRESSED,RERANKER_WITHIN_DOCUMENT_MISORDERING,HIGH,NO_EVIDENCE_OF_BOUNDARY_FAILURE,LABEL_SUPPORTED,Treat this as a reranker regression and avoid ...
2,RAG-EVAL-008,REGRESSED,RERANKER_NEAR_NEIGHBOUR_MISORDERING,HIGH,NO_EVIDENCE_OF_BOUNDARY_FAILURE,LABEL_SUPPORTED,Prefer the semantic baseline as the default co...
3,RAG-EVAL-009,IMPROVED,CROSS_DOCUMENT_SEMANTIC_COMPETITION_CORRECTED,HIGH,NO_EVIDENCE_OF_BOUNDARY_FAILURE,LABEL_SUPPORTED,Retain as evidence that reranking can correct ...
4,RAG-EVAL-012,PERSISTENT_TOP_3_MISS,SEMANTIC_TOP_3_RECALL_FAILURE,MEDIUM,NO_DIRECT_EVIDENCE_OF_BOUNDARY_FAILURE,LABEL_SUPPORTED,Do not change chunking based on one query. Rec...


## 8. Aggregate Interpretation and Retrieval Recommendation

Summarise the query-level findings and select the final retrieval configuration based on the frozen evaluation evidence.

The recommendation does not remove the implemented cross-encoder reranker. It determines whether reranking should be the default retrieval path.

In [11]:
# Consolidate the aggregate metrics and diagnostic findings.
semantic_summary_row = summary_metrics.loc[
    summary_metrics["method_name"].eq(SEMANTIC_METHOD)
].iloc[0]

reranker_summary_row = summary_metrics.loc[
    summary_metrics["method_name"].eq(RERANKER_METHOD)
].iloc[0]

outcome_counts = (
    query_comparison["comparison_outcome"]
    .value_counts()
    .to_dict()
)

analysis_summary = pd.DataFrame(
    [
        {
            "evaluation_query_count": EXPECTED_QUERY_COUNT,
            "semantic_hit_at_1": float(
                semantic_summary_row["hit_at_1"]
            ),
            "semantic_hit_at_3": float(
                semantic_summary_row["hit_at_3"]
            ),
            "semantic_mrr_at_3": float(
                semantic_summary_row["mrr_at_3"]
            ),
            "reranker_hit_at_1": float(
                reranker_summary_row["hit_at_1"]
            ),
            "reranker_hit_at_3": float(
                reranker_summary_row["hit_at_3"]
            ),
            "reranker_mrr_at_3": float(
                reranker_summary_row["mrr_at_3"]
            ),
            "improved_query_count": int(
                outcome_counts.get("IMPROVED", 0)
            ),
            "regressed_query_count": int(
                outcome_counts.get("REGRESSED", 0)
            ),
            "unchanged_top_1_query_count": int(
                outcome_counts.get("UNCHANGED_TOP_1", 0)
            ),
            "unchanged_rank_query_count": int(
                outcome_counts.get("UNCHANGED_RANK", 0)
            ),
            "persistent_top_3_miss_count": int(
                outcome_counts.get(
                    "PERSISTENT_TOP_3_MISS",
                    0,
                )
            ),
            "mean_mrr_at_3_change": float(
                query_comparison["mrr_change"].mean()
            ),
            "recommended_default_method": SEMANTIC_METHOD,
            "reranker_role": "CONTROLLED_OPTIONAL_STAGE",
            "chunking_change_recommended": False,
        }
    ]
)

assert math.isclose(
    analysis_summary.loc[0, "semantic_hit_at_1"],
    analysis_summary.loc[0, "reranker_hit_at_1"],
    rel_tol=1e-9,
    abs_tol=1e-9,
)

assert math.isclose(
    analysis_summary.loc[0, "semantic_hit_at_3"],
    analysis_summary.loc[0, "reranker_hit_at_3"],
    rel_tol=1e-9,
    abs_tol=1e-9,
)

assert (
    analysis_summary.loc[0, "semantic_mrr_at_3"]
    > analysis_summary.loc[0, "reranker_mrr_at_3"]
)

classified_query_count = sum(
    int(outcome_counts.get(outcome, 0))
    for outcome in outcome_order
)

assert classified_query_count == EXPECTED_QUERY_COUNT

print("PASS: all queries are represented in the outcome summary")
print("PASS: semantic and reranker Hit@1 are equal")
print("PASS: semantic and reranker Hit@3 are equal")
print("PASS: semantic MRR@3 is higher than reranker MRR@3")
print(
    "Recommended default method:",
    analysis_summary.loc[0, "recommended_default_method"],
)
print(
    "Reranker role:",
    analysis_summary.loc[0, "reranker_role"],
)

display(analysis_summary.T.rename(columns={0: "value"}))

PASS: all queries are represented in the outcome summary
PASS: semantic and reranker Hit@1 are equal
PASS: semantic and reranker Hit@3 are equal
PASS: semantic MRR@3 is higher than reranker MRR@3
Recommended default method: Semantic Embedding
Reranker role: CONTROLLED_OPTIONAL_STAGE


,value
evaluation_query_count,14
semantic_hit_at_1,0.785714
semantic_hit_at_3,0.928571
semantic_mrr_at_3,0.857143
reranker_hit_at_1,0.785714
reranker_hit_at_3,0.928571
reranker_mrr_at_3,0.845238
improved_query_count,2
regressed_query_count,2
unchanged_top_1_query_count,9


## 9. Findings and Design Decision

### Findings

1. **No aggregate recall improvement**

   Semantic retrieval and semantic retrieval followed by cross-encoder reranking both achieved:

   - Hit@1: 78.6%
   - Hit@3: 92.9%

   The reranker therefore did not improve aggregate top-1 or top-3 retrieval accuracy on the frozen 14-query evaluation set.

2. **Small aggregate ranking regression**

   Semantic retrieval achieved MRR@3 of 0.857, while the reranked method achieved 0.845.

   The mean per-query MRR@3 change was approximately -0.0119.

3. **Mixed query-level behaviour**

   The reranker:

   - improved 2 queries,
   - regressed 2 queries,
   - preserved rank 1 for 9 queries,
   - left 1 persistent top-3 miss.

   The larger magnitude of the regressions outweighed the improvements in the aggregate MRR calculation.

4. **Reranking can correct semantic competition**

   The reranker corrected:

   - competition between neighbouring evidence sections for `RAG-EVAL-004`,
   - cross-document semantic competition for `RAG-EVAL-009`.

5. **Reranking can also misorder relevant sections**

   The reranker degraded:

   - the manual-review escalation-packet result for `RAG-EVAL-005`,
   - the authoritative-source-order result for `RAG-EVAL-008`.

6. **One persistent semantic recall weakness remains**

   `RAG-EVAL-012` remained outside the top 3 for semantic, hybrid, and reranked retrieval, while lexical retrieval found the labelled section at rank 3.

   The labelled section exists explicitly in the approved corpus. Therefore, this is not evidence of a missing corpus section.

7. **Candidate-stage uncertainty remains**

   The committed reranker result records retain the final top-3 chunk IDs but do not retain the complete semantic top-5 candidate list or individual cross-encoder scores.

   Consequently, the analysis cannot determine whether the relevant audit-field section:

   - was absent from the five reranker candidates, or
   - was present but reranked below the final top 3.

8. **No chunking change is justified**

   The four improved or regressed queries involve competition or misordering among complete, structurally coherent sections.

   The single persistent miss is insufficient evidence to rebuild the corpus, introduce overlapping chunks, regenerate embeddings, and invalidate the frozen evaluation baseline.

### Final retrieval decision

**Semantic embedding retrieval remains the recommended default method.**

The cross-encoder reranker remains implemented and available as a controlled optional stage because it demonstrates value on selected queries and provides the required two-stage retrieval capability. However, the frozen evaluation does not support making it the default retrieval path.

This is an evidence-based null result rather than an assumption that a more complex retrieval architecture must perform better.

In [12]:
from datetime import datetime, timezone


SUMMARY_OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_error_analysis_summary_v1.csv"
)

QUERY_COMPARISON_OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_error_analysis_query_comparison_v1.csv"
)

TAXONOMY_OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_error_taxonomy_v1.csv"
)

MANIFEST_OUTPUT_PATH = (
    RETRIEVAL_DIR
    / "retrieval_error_analysis_manifest_v1.json"
)


# Convert list-valued fields to stable JSON strings before CSV export.
query_comparison_export = query_comparison.copy()

list_columns = [
    "relevant_chunk_ids",
    "semantic_retrieved_chunk_ids",
    "reranker_retrieved_chunk_ids",
    "chunks_promoted_into_top_3",
    "chunks_removed_from_top_3",
]

for column in list_columns:
    query_comparison_export[column] = (
        query_comparison_export[column].apply(
            lambda values: json.dumps(
                values,
                ensure_ascii=False,
            )
        )
    )


analysis_summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

query_comparison_export.to_csv(
    QUERY_COMPARISON_OUTPUT_PATH,
    index=False,
)

retrieval_error_taxonomy.to_csv(
    TAXONOMY_OUTPUT_PATH,
    index=False,
)


analysis_manifest = {
    "artifact_name": "retrieval_error_analysis",
    "artifact_version": "v1",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "analysis_scope": {
        "analysis_type": "offline_post_evaluation_diagnostic",
        "evaluation_query_count": EXPECTED_QUERY_COUNT,
        "top_k": EXPECTED_TOP_K,
        "reranker_candidate_k": EXPECTED_RERANKER_CANDIDATE_K,
        "compared_methods": [
            SEMANTIC_METHOD,
            RERANKER_METHOD,
        ],
        "all_methods_used_for_diagnostics": sorted(
            EXPECTED_METHODS
        ),
        "model_or_api_calls_performed": False,
        "held_out_claim_data_used": False,
    },
    "source_evaluation": {
        "query_set": str(
            ARTIFACT_PATHS["evaluation_queries"].relative_to(
                PROJECT_ROOT
            )
        ),
        "per_query_results": str(
            ARTIFACT_PATHS["per_query_results"].relative_to(
                PROJECT_ROOT
            )
        ),
        "source_manifest": str(
            ARTIFACT_PATHS["retrieval_manifest"].relative_to(
                PROJECT_ROOT
            )
        ),
        "corpus_chunk_count": int(
            retrieval_manifest["corpus"]["chunk_count"]
        ),
        "corpus_document_count": int(
            retrieval_manifest["corpus"]["document_count"]
        ),
        "semantic_embedding_model": (
            retrieval_manifest["models"][
                "semantic_embedding_model"
            ]
        ),
        "cross_encoder_model": (
            retrieval_manifest["models"][
                "cross_encoder_model"
            ]
        ),
    },
    "headline_findings": {
        "semantic_embedding": {
            "hit_at_1": float(
                semantic_summary_row["hit_at_1"]
            ),
            "hit_at_3": float(
                semantic_summary_row["hit_at_3"]
            ),
            "mrr_at_3": float(
                semantic_summary_row["mrr_at_3"]
            ),
        },
        "semantic_cross_encoder_reranker": {
            "hit_at_1": float(
                reranker_summary_row["hit_at_1"]
            ),
            "hit_at_3": float(
                reranker_summary_row["hit_at_3"]
            ),
            "mrr_at_3": float(
                reranker_summary_row["mrr_at_3"]
            ),
        },
        "improved_query_count": int(
            outcome_counts.get("IMPROVED", 0)
        ),
        "regressed_query_count": int(
            outcome_counts.get("REGRESSED", 0)
        ),
        "unchanged_top_1_query_count": int(
            outcome_counts.get("UNCHANGED_TOP_1", 0)
        ),
        "persistent_top_3_miss_count": int(
            outcome_counts.get(
                "PERSISTENT_TOP_3_MISS",
                0,
            )
        ),
        "mean_mrr_at_3_change": float(
            query_comparison["mrr_change"].mean()
        ),
    },
    "recommendation": {
        "default_retrieval_method": SEMANTIC_METHOD,
        "reranker_role": "CONTROLLED_OPTIONAL_STAGE",
        "chunking_change_recommended": False,
        "rationale": (
            "The reranker matched semantic retrieval on Hit@1 and "
            "Hit@3 but produced a slightly lower MRR@3. Query-level "
            "analysis found two improvements, two regressions, nine "
            "unchanged top-1 results, and one persistent top-3 miss."
        ),
    },
    "limitations": [
        (
            "The frozen retrieval evaluation contains 14 queries, "
            "so family-level conclusions are directional."
        ),
        (
            "The committed reranker results retain final top-3 "
            "outputs but not the complete top-5 candidate list or "
            "individual cross-encoder scores."
        ),
        (
            "The persistent audit-field miss cannot therefore be "
            "attributed conclusively to candidate retrieval or "
            "reranker scoring."
        ),
        (
            "No corpus, chunking, embedding, or relevance-label "
            "changes were made during error analysis."
        ),
    ],
    "output_files": {
        "summary": str(
            SUMMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT)
        ),
        "query_comparison": str(
            QUERY_COMPARISON_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "error_taxonomy": str(
            TAXONOMY_OUTPUT_PATH.relative_to(PROJECT_ROOT)
        ),
        "manifest": str(
            MANIFEST_OUTPUT_PATH.relative_to(PROJECT_ROOT)
        ),
    },
}

with MANIFEST_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        analysis_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    "Saved:",
    SUMMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT),
)
print(
    "Saved:",
    QUERY_COMPARISON_OUTPUT_PATH.relative_to(
        PROJECT_ROOT
    ),
)
print(
    "Saved:",
    TAXONOMY_OUTPUT_PATH.relative_to(PROJECT_ROOT),
)
print(
    "Saved:",
    MANIFEST_OUTPUT_PATH.relative_to(PROJECT_ROOT),
)

Saved: data/evaluation/retrieval/retrieval_error_analysis_summary_v1.csv
Saved: data/evaluation/retrieval/retrieval_error_analysis_query_comparison_v1.csv
Saved: data/evaluation/retrieval/retrieval_error_taxonomy_v1.csv
Saved: data/evaluation/retrieval/retrieval_error_analysis_manifest_v1.json


## 11. Artifact Read-Back Validation

Reload the exported artifacts and verify their expected shape and key contents.

In [13]:
saved_summary = pd.read_csv(SUMMARY_OUTPUT_PATH)

saved_query_comparison = pd.read_csv(
    QUERY_COMPARISON_OUTPUT_PATH
)

saved_taxonomy = pd.read_csv(
    TAXONOMY_OUTPUT_PATH
)

with MANIFEST_OUTPUT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_manifest = json.load(file)


assert saved_summary.shape == (1, analysis_summary.shape[1])

assert len(saved_query_comparison) == EXPECTED_QUERY_COUNT

assert saved_query_comparison["query_id"].nunique() == (
    EXPECTED_QUERY_COUNT
)

assert len(saved_taxonomy) == 5

assert set(saved_taxonomy["query_id"]) == (
    expected_diagnostic_ids
)

assert saved_manifest["artifact_name"] == (
    "retrieval_error_analysis"
)

assert saved_manifest["artifact_version"] == "v1"

assert (
    saved_manifest["analysis_scope"][
        "model_or_api_calls_performed"
    ]
    is False
)

assert (
    saved_manifest["analysis_scope"][
        "held_out_claim_data_used"
    ]
    is False
)

assert (
    saved_manifest["recommendation"][
        "default_retrieval_method"
    ]
    == SEMANTIC_METHOD
)

assert (
    saved_manifest["recommendation"][
        "chunking_change_recommended"
    ]
    is False
)

print("PASS: saved summary artifact validated")
print("PASS: 14 query-comparison records validated")
print("PASS: 5 diagnostic taxonomy records validated")
print("PASS: analysis manifest validated")
print("PASS: no model or API calls recorded")
print("PASS: no held-out claim data used")

display(saved_summary)

PASS: saved summary artifact validated
PASS: 14 query-comparison records validated
PASS: 5 diagnostic taxonomy records validated
PASS: analysis manifest validated
PASS: no model or API calls recorded
PASS: no held-out claim data used


,evaluation_query_count,semantic_hit_at_1,semantic_hit_at_3,semantic_mrr_at_3,reranker_hit_at_1,reranker_hit_at_3,reranker_mrr_at_3,improved_query_count,regressed_query_count,unchanged_top_1_query_count,unchanged_rank_query_count,persistent_top_3_miss_count,mean_mrr_at_3_change,recommended_default_method,reranker_role,chunking_change_recommended
0,14,0.785714,0.928571,0.857143,0.785714,0.928571,0.845238,2,2,9,0,1,-0.011905,Semantic Embedding,CONTROLLED_OPTIONAL_STAGE,False
